In [18]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import time

# Set to store unique cleaned surnames
unique_surnames = set()

# Base URL with pagination
base_url = 'https://surnames.behindthename.com/names/usage/dutch/{}'

# Loop through the first 3 pages
for page_num in range(1, 4):
    url = base_url.format(page_num)
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    name_divs = soup.find_all('div', class_='browsename')

    for div in name_divs:
        a_tag = div.find('a')
        if a_tag:
            # Clean name: remove numbers and trim whitespace
            name_raw = a_tag.text.strip()
            name_clean = re.sub(r'\s*\d+$', '', name_raw)  
            unique_surnames.add(name_clean)

# Convert to sorted list
cleaned_list = sorted(unique_surnames)

# Create DataFrame
df_dutch = pd.DataFrame(cleaned_list, columns=["Surname"])
df_dutch['Surname'] = df_dutch['Surname'].str.lower()
# Save to CSV
df_dutch["Nationality"] = 1
df_dutch


,Surname,Nationality
0,aafjes,1
1,aaij,1
2,aakster,1
3,aaldenberg,1
4,aalders,1
...,...,...
611,zaal,1
612,zeegers,1
613,zeelen,1
614,zegers,1


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import time
import random

def scrape_surnames(usage, max_pages=10):
    """Scrapes surnames for a given usage from BehindTheName."""
    base_url = f'https://surnames.behindthename.com/names/usage/{usage}/{{}}'
    surnames = set()

    for page_num in range(1, max_pages + 1):
        url = base_url.format(page_num)
        print(f"Scraping {usage} - Page {page_num}: {url}")
        response = requests.get(url)
        if response.status_code != 200:
            print("  ⚠️ Page fetch failed or finished.")
            break

        soup = BeautifulSoup(response.text, 'html.parser')
        name_divs = soup.find_all('div', class_='browsename')
        found = 0

        for div in name_divs:
            a_tag = div.find('a')
            if a_tag:
                name_raw = a_tag.text.strip()
                name_clean = re.sub(r'\s*\d+$', '', name_raw)
                name_clean = name_clean.lower()
                if name_clean not in surnames:
                    surnames.add(name_clean)
                    found += 1

        if found == 0:
            print("  🔚 No new names found. Stopping.")
            break

        time.sleep(1.2)

    return list(surnames)


# 🧩 Configuration
full_usages = ['flemish', 'estonian', 'turkish']
sample_usages = {
    'german': 150,
    'english': 150,
    'french': 50,
    'spanish': 50,
    'italian': 50,
    'polish': 50,
    'swedish': 50,
    'danish' : 50
}

# 📦 Scrape full usages
all_names = []

for usage in full_usages:
    names = scrape_surnames(usage, max_pages=10)
    all_names.extend([[name, 0] for name in names])
    print(f"✅ {usage}: {len(names)} names added")

# 🎯 Scrape sample usages
for usage, count in sample_usages.items():
    names = scrape_surnames(usage, max_pages=10)
    if len(names) >= count:
        sampled = random.sample(names, count)
    else:
        sampled = names  # take all if not enough
        print(f"⚠️ Only {len(names)} {usage} names available (requested {count})")
    all_names.extend([[name, 0] for name in sampled])
    print(f"✅ {usage}: {len(sampled)} names sampled and added")

# 🧼 Final cleanup
df_non_dutch = pd.DataFrame(all_names, columns=["Surname", "Nationality"])
df_non_dutch.drop_duplicates(subset='Surname', inplace=True)
df_non_dutch = df_non_dutch.sample(frac=1, random_state=42).reset_index(drop=True)

# ✅ Done
print(f"\n🎉 Final df_non_dutch: {len(df_non_dutch)} unique surnames")
print(df_non_dutch.head())

# Optional: Save to file
# df_non_dutch.to_csv("df_non_dutch.csv", index=False)


Scraping flemish - Page 1: https://surnames.behindthename.com/names/usage/flemish/1
Scraping flemish - Page 2: https://surnames.behindthename.com/names/usage/flemish/2
  🔚 No new names found. Stopping.
✅ flemish: 37 names added
Scraping estonian - Page 1: https://surnames.behindthename.com/names/usage/estonian/1
Scraping estonian - Page 2: https://surnames.behindthename.com/names/usage/estonian/2
  🔚 No new names found. Stopping.
✅ estonian: 16 names added
Scraping german - Page 1: https://surnames.behindthename.com/names/usage/german/1
Scraping german - Page 2: https://surnames.behindthename.com/names/usage/german/2
Scraping german - Page 3: https://surnames.behindthename.com/names/usage/german/3
Scraping german - Page 4: https://surnames.behindthename.com/names/usage/german/4
  🔚 No new names found. Stopping.
✅ german: 150 names sampled and added
Scraping english - Page 1: https://surnames.behindthename.com/names/usage/english/1
Scraping english - Page 2: https://surnames.behindthena

In [25]:
df_non_dutch

,Surname,Nationality
0,andresen,0
1,blakeley,0
2,bösch,0
3,lucas,0
4,hofmeister,0
...,...,...
673,hertz,0
674,gerst,0
675,botterill,0
676,bustos,0


In [ ]:
df_all = pd.concat([df_dutch, df_non_dutch], ignore_index=True)

# Drop any potential duplicates (just in case)
df_all.drop_duplicates(subset='Surname', inplace=True)

# Shuffle the dataset
#df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

# Preview
print(f"✅ Final merged dataset: {len(df_all)} surnames")
print(df_all.head())

# Optional: Save to file
df_all.to_csv("final_surname_dataset.csv", index=False)


✅ Final merged dataset: 1244 surnames
      Surname  Nationality
0      aafjes            1
1        aaij            1
2     aakster            1
3  aaldenberg            1
4     aalders            1


In [27]:
df_all.to_csv("final_surname_dataset.csv", index=False)
